# VQE take-home assignment

Qiskit Fall Fest 2026, IISER Kolkata. 

In the live session we ran VQE end to end for a hydrogen **atom**. Now you will repeat the pipeline for the **H₂ molecule** and then run some experiments of your own.

**How this notebook works**
- Cells marked **TODO** are yours to fill in. Replace each `...` with code.
- A cell marked `# Check` tells you whether you are on track. It only tests the number, it is not a grader.
- Cells without a TODO are provided. Run them in order, top to bottom.
- The whole notebook takes about a minute to run once it is filled in.

**Parts**
- **Part A.** Setup and a quick recap of the H atom (provided)
- **Part B.** H₂ molecule and the reaction energy (fill in)
- **Part C.** Experiments: shots, ansatz size, optimizers (fill in)
- **Part D.** Written questions

## Part A: Setup and recap (provided)

Run all cells in this part. It is the same workflow as the live session, written compactly.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error
from qiskit_aer.primitives import EstimatorV2 as Estimator

In [ ]:
NOISY = True  # set to False for a noiseless simulator

basis_gates = ["rz", "sx", "x", "cx"]  # native gates of IBM devices

if NOISY:
    noise_model = NoiseModel(basis_gates=basis_gates)
    noise_model.add_all_qubit_quantum_error(depolarizing_error(0.002, 1), ["sx", "x"])
else:
    noise_model = None

backend = AerSimulator(noise_model=noise_model, basis_gates=basis_gates)


def make_estimator(precision):
    """Aer Estimator with a given statistical precision.
    Number of shots is about 1 / precision**2 (0.01 means about 10,000 shots)."""
    return Estimator.from_backend(backend, options={"default_precision": precision})


def cost_func(params, ansatz, hamiltonian, estimator):
    """Energy of the ansatz state, from the Estimator. Logs every call."""
    pub = (ansatz, [hamiltonian], [params])
    energy = estimator.run(pubs=[pub]).result()[0].data.evs[0]
    cost_history_dict["iters"] += 1
    cost_history_dict["cost_history"].append(energy)
    return energy


def run_vqe(ansatz_isa, ham_isa, x0, precision=0.01, method="cobyla", maxiter=20):
    """Run VQE once. Returns the scipy result and the list of energies seen."""
    global cost_history_dict
    cost_history_dict = {"iters": 0, "cost_history": []}
    est = make_estimator(precision)
    res = minimize(cost_func, x0, args=(ansatz_isa, ham_isa, est),
                   method=method, options={"maxiter": maxiter})
    return res, list(cost_history_dict["cost_history"])

In [ ]:
# H atom: Hamiltonian, ansatz, transpile, run (the live-session workflow)
H_atom = SparsePauliOp.from_list([("I", -0.2355), ("Z", 0.2355)])
h = np.linalg.eigvalsh(H_atom.to_matrix()).min()

theta, phi, lam = Parameter("θ"), Parameter("φ"), Parameter("λ")
ansatz = QuantumCircuit(1)
ansatz.rx(theta, 0)
ansatz.rz(phi, 0)
ansatz.rx(lam, 0)

pm = generate_preset_pass_manager(backend=backend, optimization_level=3)
ansatz_isa = pm.run(ansatz)
H_atom_isa = H_atom.apply_layout(layout=ansatz_isa.layout)

res_atom, hist_atom = run_vqe(ansatz_isa, H_atom_isa, x0=[1, 1, 0], precision=0.01, maxiter=10)
h_vqe = res_atom.fun
print("H atom, exact :", h)
print("H atom, VQE   :", h_vqe)

## Part B: The H₂ molecule

The one-qubit Hamiltonian of H₂ (STO-6G basis, Jordan-Wigner mapping, parity qubit reduction) at a bond distance of **0.735 Å** is

$$\hat{H}_{el} = -1.04886 \, I - 0.79674 \, Z + 0.18122 \, X$$

This is only the *electronic* energy. The total energy also includes the repulsion between the two nuclei.

In [ ]:
h2_hamiltonian = SparsePauliOp.from_list(
    [("I", -1.04886087), ("Z", -0.7967368), ("X", 0.18121804)]
)

### B1. Nuclear repulsion (TODO)

Two protons repel with energy $E = 1/R$ in atomic units, where $R$ is in **Bohr radii** and the result is in hartree. Use $1\,\text{Å} = 1.88973\,a_0$.

Compute `nuclear_repulsion` for $R = 0.735$ Å.

In [ ]:
R_angstrom = 0.735
R_bohr = ...              # TODO: convert to Bohr radii
nuclear_repulsion = ...   # TODO: 1 / R in hartree

print("Nuclear repulsion (Hartree):", nuclear_repulsion)
# Check
assert abs(nuclear_repulsion - 0.71997) < 1e-3, "Expected about 0.71997"
print("OK")

### B2. Exact answer for comparison (TODO)

Diagonalize the 2x2 matrix of `h2_hamiltonian` (hint: `h2_hamiltonian.to_matrix()` and `np.linalg.eigvalsh`) to get the exact electronic ground state energy, then add the nuclear repulsion.

In [ ]:
h2_exact_el = ...                     # TODO: lowest eigenvalue of h2_hamiltonian
h2_exact = ...                        # TODO: total energy = electronic + nuclear repulsion

print("Electronic (Hartree):", h2_exact_el)
print("Total (Hartree):     ", h2_exact)
# Check
assert abs(h2_exact - (-1.14598)) < 1e-3
print("OK")

### B3. Map the Hamiltonian to the transpiled ansatz (TODO)

We already have `ansatz_isa`, the transpiled three-parameter ansatz, and H₂ also uses one qubit. Apply the ansatz layout to `h2_hamiltonian`, exactly as we did for the H atom.

In [ ]:
h2_hamiltonian_isa = ...   # TODO
print(h2_hamiltonian_isa)

### B4. Run VQE for H₂ (TODO)

Use the helper `run_vqe(ansatz_isa, ham_isa, x0, precision, method, maxiter)`. Use the initial point `[2, 0, 0]`, `precision=0.01` and `maxiter=15`. Keep the energy history it returns, and add the nuclear repulsion to the optimized energy.

In [ ]:
x0 = [2, 0, 0]
res_h2, hist_h2 = ...      # TODO: call run_vqe
h2_vqe = ...               # TODO: res_h2.fun plus nuclear repulsion

print("Exact total energy:", h2_exact)
print("VQE total energy:  ", h2_vqe)
# Check
assert abs(h2_vqe - h2_exact) < 0.05, "VQE should land within 0.05 Hartree"
print("OK")

### B5. Plot the convergence (TODO)

Plot the energy history `hist_h2` against the iteration number, and draw a horizontal line at the exact *electronic* energy `h2_exact_el` (the history does not include the nuclear repulsion). Label the axes and add a legend.

In [ ]:
fig, ax = plt.subplots()
# TODO: plot hist_h2 and a horizontal line at h2_exact_el
...
plt.show()

### B6. Reaction energy of H + H → H₂ (TODO)

$$E_{reaction} = E_{H_2} - 2E_H$$

Compute it twice: once with the exact energies (`h2_exact`, `h`) and once with the VQE energies (`h2_vqe`, `h_vqe`). A negative value means energy is released when the bond forms.

In [ ]:
E_reaction_exact = ...   # TODO
E_reaction_vqe = ...     # TODO

print("Reaction energy, exact:", E_reaction_exact)
print("Reaction energy, VQE:  ", E_reaction_vqe)
# Check
assert abs(E_reaction_exact - (-0.2039)) < 5e-3
assert abs(E_reaction_vqe - E_reaction_exact) < 0.1
print("OK")

## Part C: Experiments

For each experiment, run it and then write one or two sentences in the markdown cell below it saying what you see and why.

### C1. How many shots do you need? (TODO)

The `precision` argument controls the shot noise: shots are about `1 / precision**2`. For each precision below, run VQE on H₂ (same setup as B4) and record the absolute error of the final *electronic* energy compared with `h2_exact_el`. Then plot error against precision on log-log axes.

In [ ]:
precisions = [0.1, 0.03, 0.01, 0.003]
errors = []
for p in precisions:
    res, _ = ...                    # TODO: run_vqe with this precision
    errors.append(...)              # TODO: abs(res.fun - h2_exact_el)

fig, ax = plt.subplots()
# TODO: log-log plot of errors vs precisions
...
plt.show()

**Your observation (C1):** _write here_. Does the error keep falling as you add shots? Results are random from run to run, so repeat the cell a couple of times. Other limits are the gate noise and the 15 optimizer steps. (Hint: set `NOISY = False` at the top and re-run.)

### C2. A bigger ansatz (TODO)

Build a 12-parameter ansatz: four repeated layers of `rx, rz, rx` on one qubit, each with its own parameter (a `ParameterVector` of length 12 is handy). Transpile it with `pm`, map the Hamiltonian, and run VQE with `x0 = np.ones(12)`, `maxiter=20`, `precision=0.022` (about 2048 shots). Do the same for the 3-parameter `ansatz_isa`, then plot both histories on the same axes.

In [ ]:
from qiskit.circuit import ParameterVector

beta = ParameterVector("beta", 12)
ansatz_big = QuantumCircuit(1)
# TODO: add four layers of rx, rz, rx using beta[...]
...

ansatz_big_isa = ...          # TODO: transpile with pm
h2_big_isa = ...              # TODO: apply layout

_, hist_small = ...           # TODO: run_vqe, 3 parameters, x0=np.ones(3)
_, hist_big = ...             # TODO: run_vqe, 12 parameters, x0=np.ones(12)

fig, ax = plt.subplots()
# TODO: plot both histories with labels
...
plt.show()

**Your observation (C2):** _write here_. Which converges faster? Is the 12-parameter ansatz able to reach a lower energy than the 3-parameter one? Why or why not, for a one-qubit problem?

### C3. Choose an optimizer (TODO)

`scipy.optimize.minimize` supports many gradient-free methods. Using the same H₂ setup and `maxiter=30`, compare `"cobyla"`, `"nelder-mead"` and `"powell"`. For each, print the final electronic energy error and the number of cost-function evaluations (`res.nfev`).

In [ ]:
for method in ["cobyla", "nelder-mead", "powell"]:
    res, _ = ...      # TODO: run_vqe with this method, x0=[2, 0, 0], maxiter=30
    print(f"{method:12s} error = {...:.4f}   evaluations = {...}")  # TODO: fill both

**Your observation (C3):** _write here_. Each evaluation is one quantum-computer call. Which method is the most economical?

## Part D: Written questions

Answer in the cells below, in a few sentences each.

**D1.** The variational principle says VQE can never go below the true ground state energy. Yet in your runs the VQE energy sometimes came out slightly *below* the exact value. How is this possible?

_Your answer:_

**D2.** Why does VQE use a classical optimizer at all? What would we do with the quantum computer alone?

_Your answer:_

**D3.** We reduced H₂ to one qubit using symmetries. Why does this matter on today's noisy devices?

_Your answer:_

**D4.** Your friend says "a bigger ansatz always gives a better energy." Give one reason it might not, using what you saw in C2.

_Your answer:_